# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [1]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [2]:
LITE_MODE = False

In [3]:
username = "ed-donner"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

README.md:   0%|          | 0.00/748 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/10.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/10.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 820,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item We

In [23]:
items[2].id

2

In [24]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [25]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [26]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [27]:
import os
import anthropic

INPUT_PRICE_PER_MTOK = 1.0
OUTPUT_PRICE_PER_MTOK = 5.0

sync_claude = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
response = sync_claude.messages.create(
    model="claude-haiku-4-5",
    max_tokens=200,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": items[0].full}],
)
summary = "\n".join(block.text for block in response.content if block.type == "text").strip()
cost = (
    response.usage.input_tokens * INPUT_PRICE_PER_MTOK / 1_000_000
    + response.usage.output_tokens * OUTPUT_PRICE_PER_MTOK / 1_000_000
)

print(summary)
print()
print(f"Input tokens: {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")
print(f"Estimated cost: {cost*100:.3f} cents")


Title: Schlage F59 Andover Interior Door Knob with Deadbolt
Category: Home Hardware
Brand: Schlage
Description: Interior half of a two-piece handleset featuring a deadbolt and knob in oil rubbed bronze finish for enhanced home security.
Details: Precision-engineered with non-handed design, requires F58 exterior component to complete the set, and includes a lifetime mechanical and finish warranty.

Input tokens: 434
Output tokens: 100
Estimated cost: 0.093 cents


In [28]:

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


### Product Description
Category: Home Decor
Brand: Schlage
Description: A stylish oil rubbed bronze interior half only handle set with a deadbolt that enhances the security and aesthetic appeal of any door.
Details: Features precision-engineered 100% solid construction for a long-lasting and durable finish.

Input tokens: 406
Output tokens: 61
Cost: 0.000 cents


In [29]:
MODEL = "claude-haiku-4-5"


In [30]:
def make_jsonl(item):
    params = {
        "model": MODEL,
        "max_tokens": 200,
        "system": SYSTEM_PROMPT,
        "messages": [{"role": "user", "content": item.full}],
    }
    line = {"custom_id": str(item.id), "params": params}
    return json.dumps(line)

In [31]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [32]:
make_jsonl(items[0])

'{"custom_id": "0", "params": {"model": "claude-haiku-4-5", "max_tokens": 200, "system": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features", "messages": [{"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\

In [33]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [34]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [35]:
import os
import anthropic

claude = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

In [36]:
with open("jsonl/0_1000.jsonl", "r") as f:
    requests = [json.loads(line) for line in f if line.strip()]

len(requests), requests[0]

(1000,
 {'custom_id': '0',
  'params': {'model': 'claude-haiku-4-5',
   'max_tokens': 200,
   'system': 'Create a concise description of a product. Respond only in this format. Do not include part numbers.\nTitle: Rewritten short precise title\nCategory: eg Electronics\nBrand: Brand name\nDescription: 1 sentence description\nDetails: 1 sentence on features',
   'messages': [{'role': 'user',
     'content': 'Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish

In [37]:
requests[0]["custom_id"]

'0'

In [44]:
response = claude.messages.batches.create(requests=requests)
response

MessageBatch(id='msgbatch_01K6PKUowhHn7jZgdjF41esU', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 5, 8, 0, 31, 48, 610759, tzinfo=datetime.timezone.utc), ended_at=None, expires_at=datetime.datetime(2026, 5, 9, 0, 31, 48, 610759, tzinfo=datetime.timezone.utc), processing_status='in_progress', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=1000, succeeded=0), results_url=None, type='message_batch')

In [49]:
result = claude.messages.batches.retrieve(response.id)
result.processing_status, result.request_counts

# result

('ended',
 MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=1000))

In [50]:
if result.processing_status != "ended":
    print("Batch is not ready yet - rerun this cell after result.processing_status is 'ended'.")
else:
    with open("jsonl/batch_results.jsonl", "w") as f:
        for batch_result in claude.messages.batches.results(response.id):
            payload = {"custom_id": batch_result.custom_id, "type": batch_result.result.type}
            if batch_result.result.type == "succeeded":
                payload["summary"] = "\n".join(
                    block.text for block in batch_result.result.message.content if block.type == "text"
                ).strip()
            f.write(json.dumps(payload))
            f.write("\n")

In [51]:
with open("jsonl/batch_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        if json_line["type"] != "succeeded":
            continue
        id = int(json_line["custom_id"])
        items[id].summary = json_line["summary"]


In [52]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [56]:
print(items[999].summary)

Title: Gas Range Hot Surface Igniter Assembly
Category: Appliances
Brand: Supplying Demand
Description: Universal replacement hot surface igniter for gas ranges that lights gas flames with 3.4-3.6 amps at 120 volts.
Details: Features a 1-1/2 inch block, 3-3/4 inch cage, and 8 inch leads with female pins for easy installation.


## I've put exactly this logic into a Batch class

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

With Claude, pricing depends on your model choice and Anthropic tier. In this notebook and the updated `Batch` helper, the default model is `claude-haiku-4-5` to keep preprocessing costs down.

But you don't need to pay anything! In the next lab, you can load my pre-processed results.

In [54]:
Batch.create(items, LITE_MODE)

Created 820 batches


In [55]:
Batch.run()

  0%|          | 0/820 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': 'Number of Message Batch requests in the middle of processing has exceeded your available limit. Please try again later or contact sales at https://claude.com/contact-sales to discuss your options for a limit increase.'}, 'request_id': 'req_011CaqqJwG4vpHtxWJ2CMbFa'}

In [ ]:
Batch.fetch()

In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
print(items[10234].summary)

In [ ]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [ ]:
username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

## And here they are!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
